In [2]:
import pandas as pd
import numpy as np

# Source

## Data Sources
- *Statistics on Waste Generation and Treatment in Korea*, Ministry of Environment  
  [https://stat.me.go.kr/portal/intro/siteMapPage.do](https://stat.me.go.kr/portal/intro/siteMapPage.do)
- Parameters calibrated from `./extdata/11.Landfill_GHG_Emissions_Calculation_Method.pdf` and IPCC Tier-1 (R) methodology.

## Implemented Input Files
- `/input/policy/korea-2035/waste/landfill_ban_cp.xml`  
- `/input/policy/korea-2035/waste/landfill_ban_ep.xml`

# Landfill Ban Policy

Methane (CH₄) emissions from municipal solid waste (MSW) landfills are calculated using the **First Order Decay (FOD)** model.  

In the *Current Policy* scenario:  
- A **ban on direct landfill** is applied to the **Seoul Metropolitan Area after 2025**.  
- The policy is expanded to the **entire country after 2030**.  
- Landfilled waste is reallocated to recycling and incineration.  
- **Effect**:  
  - 1.73 MtCO₂-eq reduction by 2030  
  - 2.23 MtCO₂-eq reduction by 2035  

In the *Enhanced Ambition* scenario:  
- The **ban policy** is applied to the **entire country after 2025**.  
- **Effect**:  
  - 2.44 MtCO₂-eq reduction by 2030  
  - 3.17 MtCO₂-eq reduction by 2035


Below are brief explantion for the FOD model and detailed calcalation of mitigation effects. Final implementation of effects are presented in `waste_implementation.ipynb`

# Brief Explanation for the FOD Model

## 1. Model Concept

* **Stock‐and‐Decay:**  Each year’s deposited biodegradable carbon “stock” decays over time, releasing methane according to a first‐order (exponential) decay process.
* **Age‐Dependent Emissions:**  Emissions at time $t$ arise not only from material newly landfilled in year $t$, but from all past deposits that are still decomposing.

---

## 2. Core Equation

For year $t$, total methane emitted is

$$
\mathrm{CH}_4(t)
= \sum_{i=0}^{t} \Bigl[ M_i \times DOC \times DOC_f \times MCF \times (1 - R) \Bigr]
  \times \bigl(e^{-k\,(t - i - 1)} - e^{-k\,(t - i)}\bigr)
  \times \frac{16}{12}
$$

where:

| Symbol                                  | Meaning                                                                    |
| :-------------------------------------- | :------------------------------------------------------------------------- |
| $M_i$                                   | Mass of waste landfilled in year $i$ (t)                                   |
| $DOC$                                   | Fraction of waste that is degradable organic carbon (t C / t waste)        |
| $DOC_f$                                 | Fraction of that organic carbon that is actually degradable                |
| $MCF$                                   | Methane correction factor (fraction of degradable carbon that becomes CH₄) |
| $R$                                     | Methane recovery rate (fraction captured by gas‐collection systems)        |
| $k$                                     | Decay constant, $k = \ln2 / t_{1/2}$                                       |
| $\bigl(e^{-k(\,)\!}-e^{-k(\,)\!}\bigr)$ | Fraction of that year’s stock that decays between $t-1$ and $t$            |
| $\tfrac{16}{12}$                        | Molecular weight ratio C → CH₄                                             |

---

## 3. Key Parameters

| Parameter | Typical Tier-1 Default | Description                                      |
| :-------- | :--------------------: | :----------------------------------------------- |
| $DOC$     |          0.15          | Fraction of waste mass as organic carbon         |
| $DOC_f$   |          0.50          | Fraction of that carbon that can decompose       |
| $MCF$     |          0.80          | Fraction of degradable carbon converted to CH₄   |
| $R$       |          0.20          | Fraction of CH₄ captured by landfill gas systems |
| $t_{1/2}$ |        10 years        | Half-life of decomposable carbon stock           |

*For a more accurate (Tier-2) approach, each of these can be adjusted using country-specific waste composition and landfill management data.*

---

## 4. Model Workflow

1. **Data preparation**

   * Annual landfill inputs $M_i$.
   * Choice of DOC, DOC\_f, MCF, R, and $t_{1/2}$.
2. **Compute decay constant**
   $\;k = \ln(2) / t_{1/2}.$
3. **Loop over years**

   * For each target year $t$, sum contributions from all past $M_i$ weighted by the decay fraction for age $(t - i)$.
4. **Convert to CO₂-equivalents**
   $\mathrm{CO_2eq}(t) = \mathrm{CH_4}(t) \times GWP_{100}$ (commonly $GWP=28$ for CH₄).

---

## 5. Advantages & Limitations

**Advantages**

* Captures the time‐lagged release of methane from past landfilled waste.
* Predicts a realistic peak and decline in emissions as old waste finishes decomposing.

**Limitations**

* Requires good estimates of decay half-life and degradability parameters.
* Simple first‐order kinetics may not capture complex microbiological or environmental controls.


# Calculation of Effects

In [3]:
DOC, DOC_f, MCF, R = 0.14, 0.5, 1.0, 0.2
k = 0.09
conv = 16/12
GWP_CH4, GWP_N2O = 28, 265

In [4]:
df = pd.read_excel('../resources/korea_msw_1996_2019.xlsx').set_index('year') * 365
df = df.reset_index()
df

,year,total,landfill,inc,recycle,Seoul (landfill),Incheon (landfill),Gyeong-gi (landfill)
0,1996,15411431.5,10609090.0,784348.5,4017993.0,2559745.0,638385.0,1571434.5
1,1997,14720596.0,9485036.0,979733.0,4255827.0,2188065.5,570750.5,1447261.5
2,1998,13930298.0,7966344.0,1208478.5,4755475.5,1865697.5,497349.0,1090364.5
3,1999,14114185.0,7446876.0,1436822.5,5230486.5,1860003.5,458367.0,923413.5
4,2000,14375233.0,6919049.5,1726705.5,5729478.0,1727326.0,448329.5,747702.5
5,2001,14880393.0,6408487.5,2156602.5,6315303.0,1726523.0,385841.5,605900.0
6,2002,15615758.5,6390420.0,2415971.5,6809367.0,1685460.5,307950.5,613893.5
7,2003,15665106.5,6280117.0,2467984.0,6917005.5,1610270.5,296818.0,632800.5
8,2004,15175970.0,5458429.0,2381406.0,7336135.0,1452700.0,232943.0,549179.0
9,2005,14806772.5,4065881.0,2530800.5,8210091.0,924545.0,201151.5,394419.0


In [5]:
# 2) 처리 비율 계산 및 2020-2035 프로젝션 (2015-2019 평균 유지)
df["p_landfill"] = df["landfill"] / df["total"]
df["p_inc"] = df["inc"] / df["total"]
df["p_recycle"] = df["recycle"] / df["total"]

mask = df["year"].between(2015, 2019)
avg_p_landfill = df.loc[mask, "p_landfill"].mean()
avg_p_inc = df.loc[mask, "p_inc"].mean()
avg_p_recycle = df.loc[mask, 'p_recycle'].mean()

proj_years = list(range(2020, 2036))
total_2019 = df.loc[df["year"] == 2019, "total"].values[0]
df_proj = pd.DataFrame({
    "year": proj_years,
    "total": total_2019,
    "p_landfill": avg_p_landfill,
    "p_inc": avg_p_inc,
    "p_recycle": avg_p_recycle,
})
df_proj["landfill"] = df_proj["total"] * df_proj["p_landfill"]
df_proj["inc"] = df_proj["total"] * df_proj["p_inc"]
df_proj["recycle"] = df_proj["total"] * df_proj["p_recycle"]

# Combine historical + projection
df_all = pd.concat([
    df[["year", "total", "landfill", "inc", 'recycle', "p_landfill", "p_inc", 'p_recycle']]
      .rename(columns={"inc": "inc"}),
    df_proj
], ignore_index=True)
df_all

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle
0,1996,15411431.5,1.060909e+07,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715
1,1997,14720596.0,9.485036e+06,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107
2,1998,13930298.0,7.966344e+06,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376
3,1999,14114185.0,7.446876e+06,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584
4,2000,14375233.0,6.919050e+06,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566
5,2001,14880393.0,6.408488e+06,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404
6,2002,15615758.5,6.390420e+06,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057
7,2003,15665106.5,6.280117e+06,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555
8,2004,15175970.0,5.458429e+06,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405
9,2005,14806772.5,4.065881e+06,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482


In [6]:
conv = 16/12
GWP_CH4, GWP_N2O = 28, 265

ch4_fod = []
for idx, row in df_all.iterrows():
    t = row["year"]
    emitted = 0.0
    for j, past in df_all.iloc[:idx+1].iterrows():
        age = t - past["year"]
        decay = np.exp(-k * (age - 1)) - np.exp(-k * age)
        stock = past["landfill"] * DOC * DOC_f * MCF * (1 - R)
        emitted += stock * decay
    ch4_fod.append(emitted * conv)
df_all["CH4_FOD_t"] = ch4_fod
df_all["CO2eq_landfill_t"] = df_all["CH4_FOD_t"] * GWP_CH4

EF_CO2_inc, EF_N2O_inc = 1.0, 0.001
df_all["CO2_inc_t"] = df_all["inc"] * EF_CO2_inc
df_all["N2O_inc_t"] = df_all["inc"] * EF_N2O_inc
df_all["CO2eq_inc_t"] = df_all["CO2_inc_t"] + df_all["N2O_inc_t"] * GWP_N2O

df_all["CO2eq_total_t"] = df_all["CO2eq_landfill_t"] + df_all["CO2eq_inc_t"]

df_all

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle,CH4_FOD_t,CO2eq_landfill_t,CO2_inc_t,N2O_inc_t,CO2eq_inc_t,CO2eq_total_t
0,1996,15411431.5,1.060909e+07,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715,74599.724380,2.088792e+06,7.843485e+05,784.348500,9.922009e+05,3.080993e+06
1,1997,14720596.0,9.485036e+06,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107,134874.751041,3.776493e+06,9.797330e+05,979.733000,1.239362e+06,5.015855e+06
2,1998,13930298.0,7.966344e+06,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376,179283.021665,5.019925e+06,1.208478e+06,1208.478500,1.528725e+06,6.548650e+06
3,1999,14114185.0,7.446876e+06,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584,216216.392404,6.054059e+06,1.436822e+06,1436.822500,1.817580e+06,7.871639e+06
4,2000,14375233.0,6.919050e+06,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566,246259.444735,6.895264e+06,1.726706e+06,1726.705500,2.184282e+06,9.079547e+06
5,2001,14880393.0,6.408488e+06,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404,270126.618633,7.563545e+06,2.156602e+06,2156.602500,2.728102e+06,1.029165e+07
6,2002,15615758.5,6.390420e+06,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057,291812.528288,8.170751e+06,2.415972e+06,2415.971500,3.056204e+06,1.122695e+07
7,2003,15665106.5,6.280117e+06,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555,310856.342020,8.703978e+06,2.467984e+06,2467.984000,3.122000e+06,1.182598e+07
8,2004,15175970.0,5.458429e+06,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405,322483.230323,9.029530e+06,2.381406e+06,2381.406000,3.012479e+06,1.204201e+07
9,2005,14806772.5,4.065881e+06,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482,323317.454419,9.052889e+06,2.530800e+06,2530.800500,3.201463e+06,1.225435e+07


In [7]:
proj_years = list(range(2020, 2026))
df_proj = pd.DataFrame({
    "year": proj_years,
    "total": total_2019,
    "p_landfill": avg_p_landfill,
    "p_inc": avg_p_inc,
    "p_recycle": avg_p_recycle,
})
df_proj["landfill"] = df_proj["total"] * df_proj["p_landfill"]
df_proj["inc"] = df_proj["total"] * df_proj["p_inc"]
df_proj["recycle"] = df_proj["total"] * df_proj["p_recycle"]

In [8]:
df['Seoul Metropolitan Area share'] = (df['Seoul (landfill)'] + df['Incheon (landfill)'] + df['Gyeong-gi (landfill)']) / df['landfill']
avg_s_sma = df.loc[mask, 'Seoul Metropolitan Area share'].mean()
avg_s_sma

np.float64(0.2876091925944003)

In [9]:
proj_years = list(range(2026, 2031))
df_proj_cp_p1 = pd.DataFrame({
    "year": proj_years,
    "total": total_2019,
    "p_landfill": avg_p_landfill * (1-avg_s_sma),
    "p_inc": avg_p_inc + avg_p_landfill * avg_s_sma * (avg_p_inc / (avg_p_inc + avg_p_recycle)),
    "p_recycle": avg_p_recycle + avg_p_landfill * avg_s_sma * (avg_p_recycle / (avg_p_inc + avg_p_recycle)),
})
df_proj_cp_p1["landfill"] = df_proj_cp_p1["total"] * df_proj_cp_p1["p_landfill"]
df_proj_cp_p1["inc"] = df_proj_cp_p1["total"] * df_proj_cp_p1["p_inc"]
df_proj_cp_p1["recycle"] = df_proj_cp_p1["total"] * df_proj_cp_p1["p_recycle"]

In [10]:
proj_years = list(range(2031, 2036))
df_proj_cp_p2 = pd.DataFrame({
    "year": proj_years,
    "total": total_2019,
    "p_landfill": 0,
    "p_inc": avg_p_inc + avg_p_landfill * (avg_p_inc / (avg_p_inc + avg_p_recycle)),
    "p_recycle": avg_p_recycle + avg_p_landfill * (avg_p_recycle / (avg_p_inc + avg_p_recycle)),
})
df_proj_cp_p2["landfill"] = df_proj_cp_p2["total"] * df_proj_cp_p2["p_landfill"]
df_proj_cp_p2["inc"] = df_proj_cp_p2["total"] * df_proj_cp_p2["p_inc"]
df_proj_cp_p2["recycle"] = df_proj_cp_p2["total"] * df_proj_cp_p2["p_recycle"]

In [11]:
df_proj_cp = pd.concat([df_proj_cp_p1, df_proj_cp_p2])
df_proj_cp

,year,total,p_landfill,p_inc,p_recycle,landfill,inc,recycle
0,2026,16757916.5,0.100427,0.286711,0.612122,1.682945e+06,4.804679e+06,1.025788e+07
1,2027,16757916.5,0.100427,0.286711,0.612122,1.682945e+06,4.804679e+06,1.025788e+07
2,2028,16757916.5,0.100427,0.286711,0.612122,1.682945e+06,4.804679e+06,1.025788e+07
3,2029,16757916.5,0.100427,0.286711,0.612122,1.682945e+06,4.804679e+06,1.025788e+07
4,2030,16757916.5,0.100427,0.286711,0.612122,1.682945e+06,4.804679e+06,1.025788e+07
0,2031,16757916.5,0.000000,0.318745,0.680514,0.000000e+00,5.341507e+06,1.140400e+07
1,2032,16757916.5,0.000000,0.318745,0.680514,0.000000e+00,5.341507e+06,1.140400e+07
2,2033,16757916.5,0.000000,0.318745,0.680514,0.000000e+00,5.341507e+06,1.140400e+07
3,2034,16757916.5,0.000000,0.318745,0.680514,0.000000e+00,5.341507e+06,1.140400e+07
4,2035,16757916.5,0.000000,0.318745,0.680514,0.000000e+00,5.341507e+06,1.140400e+07


In [12]:
# Combine historical + projection
df_cp = pd.concat([
    df[["year", "total", "landfill", "inc", 'recycle', "p_landfill", "p_inc", 'p_recycle']]
      .rename(columns={"inc": "inc"}),
    df_proj_cp
], ignore_index=True)
df_cp

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle
0,1996,15411431.5,1.060909e+07,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715
1,1997,14720596.0,9.485036e+06,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107
2,1998,13930298.0,7.966344e+06,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376
3,1999,14114185.0,7.446876e+06,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584
4,2000,14375233.0,6.919050e+06,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566
5,2001,14880393.0,6.408488e+06,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404
6,2002,15615758.5,6.390420e+06,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057
7,2003,15665106.5,6.280117e+06,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555
8,2004,15175970.0,5.458429e+06,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405
9,2005,14806772.5,4.065881e+06,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482


In [13]:
ch4_fod = []
for idx, row in df_cp.iterrows():
    t = row["year"]
    emitted = 0.0
    for j, past in df_cp.iloc[:idx+1].iterrows():
        age = t - past["year"]
        decay = np.exp(-k * (age - 1)) - np.exp(-k * age)
        stock = past["landfill"] * DOC * DOC_f * MCF * (1 - R)
        emitted += stock * decay
    ch4_fod.append(emitted * conv)
df_cp["CH4_FOD_t"] = ch4_fod
df_cp["CO2eq_landfill_t"] = df_cp["CH4_FOD_t"] * GWP_CH4

EF_CO2_inc, EF_N2O_inc = 1.0, 0.001
df_cp["CO2_inc_t"] = df_cp["inc"] * EF_CO2_inc
df_cp["N2O_inc_t"] = df_cp["inc"] * EF_N2O_inc
df_cp["CO2eq_inc_t"] = df_cp["CO2_inc_t"] + df_cp["N2O_inc_t"] * GWP_N2O

df_cp["CO2eq_total_t"] = df_cp["CO2eq_landfill_t"] + df_cp["CO2eq_inc_t"]

df_cp

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle,CH4_FOD_t,CO2eq_landfill_t,CO2_inc_t,N2O_inc_t,CO2eq_inc_t,CO2eq_total_t
0,1996,15411431.5,1.060909e+07,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715,74599.724380,2.088792e+06,7.843485e+05,784.348500,9.922009e+05,3.080993e+06
1,1997,14720596.0,9.485036e+06,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107,134874.751041,3.776493e+06,9.797330e+05,979.733000,1.239362e+06,5.015855e+06
2,1998,13930298.0,7.966344e+06,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376,179283.021665,5.019925e+06,1.208478e+06,1208.478500,1.528725e+06,6.548650e+06
3,1999,14114185.0,7.446876e+06,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584,216216.392404,6.054059e+06,1.436822e+06,1436.822500,1.817580e+06,7.871639e+06
4,2000,14375233.0,6.919050e+06,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566,246259.444735,6.895264e+06,1.726706e+06,1726.705500,2.184282e+06,9.079547e+06
5,2001,14880393.0,6.408488e+06,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404,270126.618633,7.563545e+06,2.156602e+06,2156.602500,2.728102e+06,1.029165e+07
6,2002,15615758.5,6.390420e+06,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057,291812.528288,8.170751e+06,2.415972e+06,2415.971500,3.056204e+06,1.122695e+07
7,2003,15665106.5,6.280117e+06,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555,310856.342020,8.703978e+06,2.467984e+06,2467.984000,3.122000e+06,1.182598e+07
8,2004,15175970.0,5.458429e+06,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405,322483.230323,9.029530e+06,2.381406e+06,2381.406000,3.012479e+06,1.204201e+07
9,2005,14806772.5,4.065881e+06,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482,323317.454419,9.052889e+06,2.530800e+06,2530.800500,3.201463e+06,1.225435e+07


In [14]:
1.168174e+07 - 9.954912e+06

1726828.0

In [15]:
1.151001e+07- 9.229087e+06

2280923.0

In [16]:
proj_years = list(range(2026, 2036))
df_proj_ep = pd.DataFrame({
    "year": proj_years,
    "total": total_2019,
    "p_landfill": 0,
    "p_inc": avg_p_inc + avg_p_landfill * (avg_p_inc / (avg_p_inc + avg_p_recycle)),
    "p_recycle": avg_p_recycle + avg_p_landfill * (avg_p_recycle / (avg_p_inc + avg_p_recycle)),
})
df_proj_ep["landfill"] = df_proj_ep["total"] * df_proj_ep["p_landfill"]
df_proj_ep["inc"] = df_proj_ep["total"] * df_proj_ep["p_inc"]
df_proj_ep["recycle"] = df_proj_ep["total"] * df_proj_ep["p_recycle"]

In [17]:
df_ep = pd.concat([
    df[["year", "total", "landfill", "inc", 'recycle', "p_landfill", "p_inc", 'p_recycle']]
      .rename(columns={"inc": "inc"}),
    df_proj_ep
], ignore_index=True)
df_ep

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle
0,1996,15411431.5,10609090.0,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715
1,1997,14720596.0,9485036.0,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107
2,1998,13930298.0,7966344.0,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376
3,1999,14114185.0,7446876.0,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584
4,2000,14375233.0,6919049.5,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566
5,2001,14880393.0,6408487.5,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404
6,2002,15615758.5,6390420.0,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057
7,2003,15665106.5,6280117.0,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555
8,2004,15175970.0,5458429.0,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405
9,2005,14806772.5,4065881.0,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482


In [18]:
conv = 16/12
GWP_CH4, GWP_N2O = 28, 265

ch4_fod = []
for idx, row in df_ep.iterrows():
    t = row["year"]
    emitted = 0.0
    for j, past in df_ep.iloc[:idx+1].iterrows():
        age = t - past["year"]
        decay = np.exp(-k * (age - 1)) - np.exp(-k * age)
        stock = past["landfill"] * DOC * DOC_f * MCF * (1 - R)
        emitted += stock * decay
    ch4_fod.append(emitted * conv)
df_ep["CH4_FOD_t"] = ch4_fod
df_ep["CO2eq_landfill_t"] = df_ep["CH4_FOD_t"] * GWP_CH4

EF_CO2_inc, EF_N2O_inc = 1.0, 0.001
df_ep["CO2_inc_t"] = df_ep["inc"] * EF_CO2_inc
df_ep["N2O_inc_t"] = df_ep["inc"] * EF_N2O_inc
df_ep["CO2eq_inc_t"] = df_ep["CO2_inc_t"] + df_ep["N2O_inc_t"] * GWP_N2O

df_ep["CO2eq_total_t"] = df_ep["CO2eq_landfill_t"] + df_ep["CO2eq_inc_t"]

df_ep

,year,total,landfill,inc,recycle,p_landfill,p_inc,p_recycle,CH4_FOD_t,CO2eq_landfill_t,CO2_inc_t,N2O_inc_t,CO2eq_inc_t,CO2eq_total_t
0,1996,15411431.5,10609090.0,7.843485e+05,4.017993e+06,0.688391,0.050894,0.260715,74599.724380,2.088792e+06,7.843485e+05,784.348500,9.922009e+05,3.080993e+06
1,1997,14720596.0,9485036.0,9.797330e+05,4.255827e+06,0.644338,0.066555,0.289107,134874.751041,3.776493e+06,9.797330e+05,979.733000,1.239362e+06,5.015855e+06
2,1998,13930298.0,7966344.0,1.208478e+06,4.755476e+06,0.571872,0.086752,0.341376,179283.021665,5.019925e+06,1.208478e+06,1208.478500,1.528725e+06,6.548650e+06
3,1999,14114185.0,7446876.0,1.436822e+06,5.230486e+06,0.527616,0.101800,0.370584,216216.392404,6.054059e+06,1.436822e+06,1436.822500,1.817580e+06,7.871639e+06
4,2000,14375233.0,6919049.5,1.726706e+06,5.729478e+06,0.481317,0.120117,0.398566,246259.444735,6.895264e+06,1.726706e+06,1726.705500,2.184282e+06,9.079547e+06
5,2001,14880393.0,6408487.5,2.156602e+06,6.315303e+06,0.430667,0.144929,0.424404,270126.618633,7.563545e+06,2.156602e+06,2156.602500,2.728102e+06,1.029165e+07
6,2002,15615758.5,6390420.0,2.415972e+06,6.809367e+06,0.409229,0.154714,0.436057,291812.528288,8.170751e+06,2.415972e+06,2415.971500,3.056204e+06,1.122695e+07
7,2003,15665106.5,6280117.0,2.467984e+06,6.917006e+06,0.400898,0.157547,0.441555,310856.342020,8.703978e+06,2.467984e+06,2467.984000,3.122000e+06,1.182598e+07
8,2004,15175970.0,5458429.0,2.381406e+06,7.336135e+06,0.359676,0.156920,0.483405,322483.230323,9.029530e+06,2.381406e+06,2381.406000,3.012479e+06,1.204201e+07
9,2005,14806772.5,4065881.0,2.530800e+06,8.210091e+06,0.274596,0.170922,0.554482,323317.454419,9.052889e+06,2.530800e+06,2530.800500,3.201463e+06,1.225435e+07


In [19]:
1.168174e+07 - 9.238931e+06

2442809.0

In [20]:
1.151001e+07- 8.339551e+06

3170459.0